In [1]:
from datasets import load_dataset, Dataset
import re
import pandas as pd

In [2]:
data = load_dataset("TheFinAI/jp-intent-commitment-scale", split="train",trust_remote_code=True)

In [3]:
data_df = data.to_pandas()

In [4]:
data_df

,id,query,answer,text,choices,gold
0,0,Analyze corporate Q&A and identify the commitm...,weak or qualified commitment,Question: 中長期 ROE 目標を 12%程度とした背景を教えてほしい。\nAnsw...,"[strong commitment, weak or qualified commitme...",1
1,1,Analyze corporate Q&A and identify the commitm...,neutral or hedged,Question: 中長期 ROE 目標は米銀主要行に追いつくことを想定しているのか。\nA...,"[strong commitment, weak or qualified commitme...",2
2,2,Analyze corporate Q&A and identify the commitm...,weak or qualified commitment,Question: 中長期ROE目標の達成時期はいつ頃を想定しているのか。また、その際には ...,"[strong commitment, weak or qualified commitme...",1
3,3,Analyze corporate Q&A and identify the commitm...,weak or qualified commitment,Question: 25 年度の業績目標の背景を教えてほしい。また、通商政策による影響で環境...,"[strong commitment, weak or qualified commitme...",1
4,4,Analyze corporate Q&A and identify the commitm...,weak or qualified commitment,Question: 日銀の金融政策の見通しと、円債の積み増しのタイミングについて教えてほしい...,"[strong commitment, weak or qualified commitme...",1
...,...,...,...,...,...,...
89,89,Analyze corporate Q&A and identify the commitm...,weak or qualified commitment,Question: 経戦 2027 の建付けでは、Enhance（「磨く」）に ASEAN ...,"[strong commitment, weak or qualified commitme...",1
90,90,Analyze corporate Q&A and identify the commitm...,weak refusal,Question: 不安感を解消したいという趣旨での質問となるが、2025 年度業績見通しの...,"[strong commitment, weak or qualified commitme...",3
91,91,Analyze corporate Q&A and identify the commitm...,weak refusal,"Question: 今回の 2025 年度連結純利益の見通し 7,000 億円について、もと...","[strong commitment, weak or qualified commitme...",3
92,92,Analyze corporate Q&A and identify the commitm...,neutral or hedged,Question: モビリティグループについて、2024 年度の連結純利益の実績が一過性損益...,"[strong commitment, weak or qualified commitme...",2


In [5]:
text2label = {
    "strong commitment": "+2",
    "weak or qualified commitment": "+1",
    "neutral or hedged": "0",
    "weak refusal": "-1",
    "strong refusal": "-2"
}

In [6]:
def extract_qa(text: str):
    """
    Extract Question and Answer from a single QA block.
    
    Expected format:
    Question: ...
    Answer: ...
    """
    pattern = r"Question:\s*(.*?)\s*Answer:\s*(.*)"
    match = re.search(pattern, text, flags=re.DOTALL)

    if not match:
        return None, None
    
    question = match.group(1).strip()
    answer = match.group(2).strip()

    return question, answer

In [7]:
def get_query(question, response):

    task1 = f"""You are a Japanese financial expert fluent in Japanese business communication.
Given a financial question and the corresponding company response (both in Japanese),
your task is to determine the company’s underlying intent level and output exactly one label from the following set: {{"+2", "+1", "0", "-1", "-2"}}, 
The label meanings are provided for explanation only (DO NOT output these texts):
    "+2" : "Strong Commitment"
    "+1" : "Weak or Qualified Commitment"
     "0" : "Neutral or Hedged Intent"
    "-1" : "Weak Refusal"
    "-2" : "Strong Refusal"
    
Financial Question: {question}
Company Response: {response}

Directly output the chosen label, and do not provide any explanation.
Answer:
"""

    return task1

In [10]:
querys = []
answers = []
for i in range(len(data_df)):
    answer = data_df.at[i,"answer"]
    print(i)
    print(answer)
    answer = text2label[answer]
    
    text = data_df.at[i, "text"]
    question, response = extract_qa(text)
    query = get_query(question, response)

    querys.append(query)
    answers.append(answer)

0
weak or qualified commitment
1
neutral or hedged
2
weak or qualified commitment
3
weak or qualified commitment
4
weak or qualified commitment
5
weak or qualified commitment
6
weak or qualified commitment
7
weak or qualified commitment
8
weak or qualified commitment
9
weak or qualified commitment
10
weak or qualified commitment
11
weak or qualified commitment
12
weak or qualified commitment
13
weak or qualified commitment
14
neutral or hedged
15
weak or qualified commitment
16
weak or qualified commitment
17
strong commitment
18
strong commitment
19
weak or qualified commitment
20
strong commitment
21
strong commitment
22
weak or qualified commitment
23
weak or qualified commitment
24
strong commitment
25
weak or qualified commitment
26
weak or qualified commitment
27
weak or qualified commitment
28
weak or qualified commitment
29
weak or qualified commitment
30
weak or qualified commitment
31
weak or qualified commitment
32
weak or qualified commitment
33
neutral or hedged
34
neutral

In [11]:
new_df = pd.DataFrame({"query":querys, "answer":answers})

In [12]:
new_df.tail(40)

,query,answer
54,You are a Japanese financial expert fluent in ...,+1
55,You are a Japanese financial expert fluent in ...,+1
56,You are a Japanese financial expert fluent in ...,0
57,You are a Japanese financial expert fluent in ...,0
58,You are a Japanese financial expert fluent in ...,+1
59,You are a Japanese financial expert fluent in ...,+1
60,You are a Japanese financial expert fluent in ...,+1
61,You are a Japanese financial expert fluent in ...,+1
62,You are a Japanese financial expert fluent in ...,+1
63,You are a Japanese financial expert fluent in ...,+1


In [13]:
new_df.at[0,"query"]

'You are a Japanese financial expert fluent in Japanese business communication.\nGiven a financial question and the corresponding company response (both in Japanese),\nyour task is to determine the company’s underlying intent level and output exactly one label from the following set: {"+2", "+1", "0", "-1", "-2"}, \nThe label meanings are provided for explanation only (DO NOT output these texts):\n    "+2" : "Strong Commitment"\n    "+1" : "Weak or Qualified Commitment"\n     "0" : "Neutral or Hedged Intent"\n    "-1" : "Weak Refusal"\n    "-2" : "Strong Refusal"\n    \nFinancial Question: 中長期 ROE 目標を 12%程度とした背景を教えてほしい。\nCompany Response: 中長期 ROE 目標は、政策保有株式の売却益がなくなり、本邦の政策金利が一定の水 準まで上昇して留まった状況を前提としており、オーガニック・インオーガニック戦略や 自己株式取得の効果なども織り込んだ上でめざす水準である。試算の前提次第だが、 その他の目線では、例えば、株価 3,100 円、時価総額 30 兆円程度も視野に入るのでは ないかと考えている。 前中計では、資本市場の評価を得るために最低限必用な水準として ROE 目標を 7.5% とした。当初は意欲的な目標と考えていたが、マイナス金利環境下において収益の多 様化を進めるため、リスクリターンの向上や手数料収益の拡大、デジタルや AM/IS 領 域での投資に取り組んできた。その結果、金利が上がり始めた現在では、10%が視野に 入るまで改善して

In [14]:
from huggingface_hub import HfApi

In [15]:
hf_dataset = Dataset.from_pandas(new_df, preserve_index=True)
hf_dataset = hf_dataset.rename_column("__index_level_0__", "id")

In [16]:
dataset_repo = "TheFinAI/JF-ICR"

In [17]:
hf_dataset.push_to_hub(dataset_repo,split="test",private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/331 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/TheFinAI/JF-ICR/commit/8caebc1cb676364bf0ccd3d65e62f9a446df0ae0', commit_message='Upload dataset', commit_description='', oid='8caebc1cb676364bf0ccd3d65e62f9a446df0ae0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/TheFinAI/JF-ICR', endpoint='https://huggingface.co', repo_type='dataset', repo_id='TheFinAI/JF-ICR'), pr_revision=None, pr_num=None)